In [1]:
import pandas as pd

In [9]:
line_df = pd.read_excel(r"C:\Users\kulareddy\Downloads\New freq load\SubscriptionLine_04012026.xlsx")

In [4]:
account_df=pd.read_csv(r"C:\Working transformation and lookup\Reports\Account-2_24_2026.csv")

C:\Users\kulareddy\AppData\Local\Temp\ipykernel_40808\3559953654.py:1: DtypeWarning: Columns (19,66) have mixed types. Specify dtype option on import or set low_memory=False.
  account_df=pd.read_csv(r"C:\Working transformation and lookup\Reports\Account-2_24_2026.csv")


In [5]:
site_df=pd.read_csv(r"C:\Working transformation and lookup\Reports\CVG_Site__c-2_24_2026.csv")

In [6]:
oracle_df=pd.read_excel(r"C:\Working transformation and lookup\Reports\Convergint Oracle Customer Detailed Dev3 20260220 01.xlsx")

In [10]:
def normalize_key(series):
    s = series.astype("string")

    s = (
        s.str.replace("\u00A0", "", regex=False)
         .str.replace(r"[\u200B-\u200D\uFEFF]", "", regex=True)
         .str.replace(r"\r|\n|\t", "", regex=True)
         .str.strip()
         .str.replace(r"\.0$", "", regex=True)
         .str.replace(r"\s+", "", regex=True)
    )

    s = s.replace({
        "": pd.NA,
        "nan": pd.NA,
        "None": pd.NA,
        "<NA>": pd.NA
    })

    return s


def build_lookup(df, key_col, value_col, normalize_value=False):
    temp = df[[key_col, value_col]].copy()
    temp[key_col] = normalize_key(temp[key_col])

    if normalize_value:
        temp[value_col] = normalize_key(temp[value_col])

    temp = (
        temp.dropna(subset=[key_col])
            .drop_duplicates(subset=[key_col], keep="first")
    )

    return temp.set_index(key_col)[value_col]


def insert_after(df, base_col, new_col, values):
    pos = df.columns.get_loc(base_col) + 1
    df.insert(pos, new_col, values)


def build_match_stats(source_clean, sfid_raw, oracle_raw, row_name):
    source_present = source_clean.notna()

    return {
        "Field": row_name,
        "Total Count": int(source_present.sum()),
        "Count Not matching for SF id": int((source_present & sfid_raw.isna()).sum()),
        "Count Not matching for Oracle id": int((source_present & sfid_raw.notna() & oracle_raw.isna()).sum())
    }


# -------------------------------------------------
# 0. Normalize source columns used for matching
# -------------------------------------------------
shipto_party_source_clean = normalize_key(line_df["ShipToPartyId"])
shipto_partysite_source_clean = normalize_key(line_df["ShipToPartySiteId"])


# -----------------------------
# 1. Build required lookups
# -----------------------------

# account_df: CVG_Legacy_ID__c -> Id
account_legacy_to_sfid = build_lookup(
    account_df,
    "CVG_Legacy_ID__c",
    "Id",
    normalize_value=True
)

# site_df: CVG_Legacy_ID__c -> Id
site_legacy_to_sfid = build_lookup(
    site_df,
    "CVG_Legacy_ID__c",
    "Id",
    normalize_value=True
)

# oracle_df lookups
# ORIG_SYS_REF -> PARTY_ID
orig_sys_ref_to_party_id = build_lookup(
    oracle_df,
    "ORIG_SYS_REF",
    "PARTY_ID",
    normalize_value=False
)

# SITE_ORIG_SYS_REF -> PARTY_SITE_ID
site_orig_sys_ref_to_party_site_id = build_lookup(
    oracle_df,
    "SITE_ORIG_SYS_REF",
    "PARTY_SITE_ID",
    normalize_value=False
)


# -----------------------------------
# 2. Create mapped raw values
# -----------------------------------

# ShipToPartyId
shipto_party_sfid_raw = shipto_party_source_clean.map(account_legacy_to_sfid)
shipto_party_oracle_raw = shipto_party_sfid_raw.map(orig_sys_ref_to_party_id)

# ShipToPartySiteId
shipto_partysite_sfid_raw = shipto_partysite_source_clean.map(site_legacy_to_sfid)
shipto_partysite_oracle_raw = shipto_partysite_sfid_raw.map(site_orig_sys_ref_to_party_site_id)


# -----------------------------------
# 3. Replace non-matches with #N/A
# -----------------------------------
shipto_party_sfid = shipto_party_sfid_raw.fillna("#N/A")
shipto_party_oracle = shipto_party_oracle_raw.fillna("#N/A")

shipto_partysite_sfid = shipto_partysite_sfid_raw.fillna("#N/A")
shipto_partysite_oracle = shipto_partysite_oracle_raw.fillna("#N/A")


# -----------------------------------
# 4. Create line_inprogress output
# -----------------------------------
line_inprogress = line_df.copy()

insert_after(line_inprogress, "ShipToPartyId", "ShipToPartyId_SFId", shipto_party_sfid)
insert_after(line_inprogress, "ShipToPartyId_SFId", "ShipToPartyId_OracleId", shipto_party_oracle)

insert_after(line_inprogress, "ShipToPartySiteId", "ShipToPartySiteId_SFId", shipto_partysite_sfid)
insert_after(line_inprogress, "ShipToPartySiteId_SFId", "ShipToPartySiteId_OracleId", shipto_partysite_oracle)


# -----------------------------------
# 5. Create line_transformed output
# -----------------------------------
line_transformed = line_df.copy()

line_transformed["ShipToPartyId"] = shipto_party_oracle
line_transformed["ShipToPartySiteId"] = shipto_partysite_oracle


# -----------------------------------
# 6. Create stats table
# -----------------------------------
stats_data = [
    build_match_stats(
        shipto_party_source_clean,
        shipto_party_sfid_raw,
        shipto_party_oracle_raw,
        "ShipToPartyId"
    ),
    build_match_stats(
        shipto_partysite_source_clean,
        shipto_partysite_sfid_raw,
        shipto_partysite_oracle_raw,
        "ShipToPartySiteId"
    )
]

stats_df = pd.DataFrame(stats_data)


# -----------------------------------
# 7. Save outputs
# -----------------------------------
with pd.ExcelWriter("line_inprogress.xlsx", engine="openpyxl") as writer:
    line_inprogress.to_excel(writer, sheet_name="line_inprogress", index=False)
    stats_df.to_excel(writer, sheet_name="stats", index=False)

line_transformed.to_excel("line_transformed.xlsx", index=False)


# -----------------------------------
# 8. Validation prints
# -----------------------------------
print("line_inprogress.xlsx created")
print("line_transformed.xlsx created")

print("\nShapes:")
print("line_df:", line_df.shape)
print("line_inprogress:", line_inprogress.shape)
print("line_transformed:", line_transformed.shape)




line_inprogress.xlsx created
line_transformed.xlsx created

Shapes:
line_df: (7960, 26)
line_inprogress: (7960, 30)
line_transformed: (7960, 26)


In [11]:
print("\nStats:")
print(stats_df)

#stats_df.to_excel("line_stats.xlsx", index=False)
#print("line_stats.xlsx created")


Stats:
               Field  Total Count  Count Not matching for SF id  \
0      ShipToPartyId         7956                           324   
1  ShipToPartySiteId         7956                          1867   

   Count Not matching for Oracle id  
0                                80  
1                                47  
